## Data Cleaning and Standardization

### Objective

The purpose of this notebook is to improve the quality of the anonymized datasets by identifying and correcting common data quality issues.

The cleaning process focuses on:

- Standardizing column names
- Removing duplicate records
- Handling missing values
- Correcting inconsistent data types
- Validating assessment scores
- Preparing the datasets for loading into the project database

The cleaned datasets generated in this notebook will serve as the final source for the SQL database.

In [3]:
import pandas as pd
import numpy as np


In [4]:
attendance = pd.read_csv("../data/processed/term1/attendance.csv")

test1 = pd.read_csv("../data/processed/term1/test1.csv")

test2 = pd.read_csv("../data/processed/term1/test2.csv")

groupwork = pd.read_csv("../data/processed/term1/groupwork.csv")

projectwork = pd.read_csv("../data/processed/term1/projectwork.csv")

classscore = pd.read_csv("../data/processed/term1/classscore.csv")

examscore = pd.read_csv("../data/processed/term1/examscore.csv")

In [4]:
def cleaning_summary(df):

    summary = pd.DataFrame({
        "Column": df.columns,
        "Data Type": df.dtypes.values,
        "Missing Values": df.isna().sum().values,
        "Duplicate Values": df.duplicated().sum(),
        "Unique Values": df.nunique().values
    })

    return summary

In [5]:
cleaning_summary(test1)

,Column,Data Type,Missing Values,Duplicate Values,Unique Values
0,ENGLISH LANGUAGE,float64,44,42,8
1,SOCIAL STUDIES,float64,44,42,7
2,REL. & MORAL EDU.,float64,43,42,8
3,MATHEMATICS,float64,43,42,7
4,INTEGRATED SCIENCE,float64,44,42,14
5,COMPUTING,float64,44,42,10
6,FRENCH,float64,45,42,10
7,GH. LANG. (AKUAPEM TWI),float64,45,42,21
8,CAREER TECHNOLOGY,float64,45,42,4
9,CREATIVE ARTS & DESIGN,float64,44,42,13


### Removing Blank Rows

The exported Excel worksheets contain several blank rows at the end of the datasets. These rows do not represent student records but are interpreted as observations with missing values when imported into pandas.

Removing these rows improves data quality by reducing unnecessary missing values and ensuring that only valid student records remain for further processing.

In [13]:
# Function to remove blank rows
def remove_blank_rows(df):

    df = df.dropna(how="all")

    if "Student_ID" in df.columns:
        df = df.dropna(subset=["Student_ID"])

    return df.reset_index(drop=True)

In [7]:
# Applying to all Datasets in term 1
attendance = remove_blank_rows(attendance)

test1 = remove_blank_rows(test1)

test2 = remove_blank_rows(test2)

groupwork = remove_blank_rows(groupwork)

projectwork = remove_blank_rows(projectwork)

classscore = remove_blank_rows(classscore)

examscore = remove_blank_rows(examscore)

In [8]:
print("Attendance:", attendance.shape)
print("Test 1:", test1.shape)
print("Test 2:", test2.shape)
print("Group Work:", groupwork.shape)
print("Project Work:", projectwork.shape)
print("Class Score:", classscore.shape)
print("Exam Score:", examscore.shape)

Attendance: (59, 2)
Test 1: (60, 11)
Test 2: (60, 11)
Group Work: (59, 11)
Project Work: (59, 11)
Class Score: (59, 22)
Exam Score: (59, 22)


In [9]:
cleaning_summary(test1)

,Column,Data Type,Missing Values,Duplicate Values,Unique Values
0,ENGLISH LANGUAGE,float64,1,0,8
1,SOCIAL STUDIES,float64,1,0,7
2,REL. & MORAL EDU.,float64,0,0,8
3,MATHEMATICS,float64,0,0,7
4,INTEGRATED SCIENCE,float64,1,0,14
5,COMPUTING,float64,1,0,10
6,FRENCH,float64,2,0,10
7,GH. LANG. (AKUAPEM TWI),float64,2,0,21
8,CAREER TECHNOLOGY,float64,2,0,4
9,CREATIVE ARTS & DESIGN,float64,1,0,13


## Correcting Data Types

Some variables were imported using data types that are not appropriate for analysis. Attendance values were stored as floating-point numbers even though attendance is recorded as whole numbers.

The data types were corrected to improve consistency and prepare the datasets for loading into the SQL database.



One student record contained a missing attendance value. Rather than removing the student or assigning a value of zero, the missing attendance was replaced with the class average. This approach preserves the student record while minimizing the impact of the missing value on subsequent analyses.

In [13]:
attendance["Attendance"] = (
    attendance["Attendance"]
    .fillna(round(attendance["Attendance"].mean()))
    .astype(int)
)

In [14]:
attendance["Attendance"].isna().sum()

np.int64(0)

In [15]:
attendance["Attendance"] = attendance["Attendance"].astype(int)

### Handle Missing Assessment Scores

Several assessment datasets contained missing student scores. To preserve all student records while maintaining a complete dataset, missing numeric values were replaced using the average score of the respective assessment column.

The average was calculated independently for each assessment column, ensuring that the imputed values reflected the overall class performance for that assessment. This approach minimizes data loss while preparing the datasets for database integration and subsequent analysis.

In [17]:
def fill_missing_scores(df):

    df = df.copy()

    # Select all numeric columns
    numeric_columns = df.select_dtypes(include="number").columns

    # Replace missing values with the column average
    for column in numeric_columns:

        average = round(df[column].mean(), 2)

        df[column] = df[column].fillna(average)

    return df

In [17]:
test1 = fill_missing_scores(test1)

test2 = fill_missing_scores(test2)

groupwork = fill_missing_scores(groupwork)

projectwork = fill_missing_scores(projectwork)

classscore = fill_missing_scores(classscore)

examscore = fill_missing_scores(examscore)

### Checkinġ for Duplicate Records

Duplicate records can affect data accuracy and lead to incorrect analyses. Each dataset was checked for duplicate rows, and any duplicates identified were removed while retaining the first occurrence.

This step ensures that each student record is represented only once within each dataset.

In [18]:
def remove_duplicates(df):

    df = df.drop_duplicates()

    return df.reset_index(drop=True)

In [5]:
attendance = remove_duplicates(attendance)

test1 = remove_duplicates(test1)

test2 = remove_duplicates(test2)

groupwork = remove_duplicates(groupwork)

projectwork = remove_duplicates(projectwork)

classscore = remove_duplicates(classscore)

examscore = remove_duplicates(examscore)

### Validating the Cleaned Data

After completing the data cleaning process, the datasets were validated to ensure that no missing values remained, duplicate records had been removed, and the data types were appropriate for analysis and database integration.

This validation step confirms that the cleaned datasets meet the required quality standards before being stored in the project database.

In [6]:
def validate_dataset(df, name):

    print(f"\n{name}")
    print("-" * 40)

    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    print("\nMissing Values")
    print(df.isna().sum())

    print("\nDuplicate Rows")
    print(df.duplicated().sum())

    print("\nData Types")
    print(df.dtypes)

In [7]:
validate_dataset(attendance, "Attendance")

validate_dataset(test1, "Test 1")

validate_dataset(test2, "Test 2")

validate_dataset(groupwork, "Group Work")

validate_dataset(projectwork, "Project Work")

validate_dataset(classscore, "Class Score")

validate_dataset(examscore, "Exam Score")


Attendance
----------------------------------------
Rows: 60
Columns: 2

Missing Values
Attendance    2
Student_ID    1
dtype: int64

Duplicate Rows
0

Data Types
Attendance    float64
Student_ID        str
dtype: object

Test 1
----------------------------------------
Rows: 61
Columns: 11

Missing Values
ENGLISH LANGUAGE           2
SOCIAL STUDIES             2
REL. & MORAL EDU.          1
MATHEMATICS                1
INTEGRATED SCIENCE         2
COMPUTING                  2
FRENCH                     3
GH. LANG. (AKUAPEM TWI)    3
CAREER TECHNOLOGY          3
CREATIVE ARTS & DESIGN     2
Student_ID                 1
dtype: int64

Duplicate Rows
0

Data Types
ENGLISH LANGUAGE           float64
SOCIAL STUDIES             float64
REL. & MORAL EDU.          float64
MATHEMATICS                float64
INTEGRATED SCIENCE         float64
COMPUTING                  float64
FRENCH                     float64
GH. LANG. (AKUAPEM TWI)    float64
CAREER TECHNOLOGY          float64
CREATIVE ARTS &

## Saving the Cleaned Datasets

The cleaned datasets were saved to the cleaned data directory. These datasets will be used for loading into the MySQL database and subsequent analysis.

In [10]:
attendance.to_csv("../data/cleaned/term1/attendance.csv", index=False)

test1.to_csv("../data/cleaned/term1/test1.csv", index=False)

test2.to_csv("../data/cleaned/term1/test2.csv", index=False)

groupwork.to_csv("../data/cleaned/term1/groupwork.csv", index=False)

projectwork.to_csv("../data/cleaned/term1/projectwork.csv", index=False)

classscore.to_csv("../data/cleaned/term1/classscore.csv", index=False)

examscore.to_csv("../data/cleaned/term1/examscore.csv", index=False)

print("Term 1 cleaned datasets saved successfully.")

Term 1 cleaned datasets saved successfully.


### Term 2 Data Cleaning

The same data cleaning procedures applied to the first academic term were repeated for the second term to ensure consistency across the entire dataset.

The cleaning process included:

- Removing blank rows
- Handling missing values
- Correcting data types
- Removing duplicate records
- Validating the cleaned datasets

Using the same workflow for each term ensures that all datasets are prepared consistently before loading them into the project database.

In [11]:
attendance_term2 = pd.read_csv("../data/processed/term2/attendance.csv")

test1_term2 = pd.read_csv("../data/processed/term2/test1.csv")

test2_term2 = pd.read_csv("../data/processed/term2/test2.csv")

groupwork_term2 = pd.read_csv("../data/processed/term2/groupwork.csv")

projectwork_term2 = pd.read_csv("../data/processed/term2/projectwork.csv")

classscore_term2 = pd.read_csv("../data/processed/term2/classscore.csv")

examscore_term2 = pd.read_csv("../data/processed/term2/examscore.csv")

In [14]:
attendance_term2 = remove_blank_rows(attendance_term2)

test1_term2 = remove_blank_rows(test1_term2)

test2_term2 = remove_blank_rows(test2_term2)

groupwork_term2 = remove_blank_rows(groupwork_term2)

projectwork_term2 = remove_blank_rows(projectwork_term2)

classscore_term2 = remove_blank_rows(classscore_term2)

examscore_term2 = remove_blank_rows(examscore_term2)

In [15]:
attendance_term2["Attendance"] = (
    attendance_term2["Attendance"]
    .fillna(round(attendance_term2["Attendance"].mean()))
    .astype(int)
)

In [19]:
test1_term2 = fill_missing_scores(test1_term2)

test2_term2 = fill_missing_scores(test2_term2)

groupwork_term2 = fill_missing_scores(groupwork_term2)

projectwork_term2 = fill_missing_scores(projectwork_term2)

classscore_term2 = fill_missing_scores(classscore_term2)

examscore_term2 = fill_missing_scores(examscore_term2)

In [20]:
attendance_term2 = remove_duplicates(attendance_term2)

test1_term2 = remove_duplicates(test1_term2)

test2_term2 = remove_duplicates(test2_term2)

groupwork_term2 = remove_duplicates(groupwork_term2)

projectwork_term2 = remove_duplicates(projectwork_term2)

classscore_term2 = remove_duplicates(classscore_term2)

examscore_term2 = remove_duplicates(examscore_term2)

In [21]:
validate_dataset(attendance_term2, "Attendance")

validate_dataset(test1_term2, "Test 1")

validate_dataset(test2_term2, "Test 2")

validate_dataset(groupwork_term2, "Group Work")

validate_dataset(projectwork_term2, "Project Work")

validate_dataset(classscore_term2, "Class Score")

validate_dataset(examscore_term2, "Exam Score")


Attendance
----------------------------------------
Rows: 57
Columns: 2

Missing Values
Attendance    0
Student_ID    0
dtype: int64

Duplicate Rows
0

Data Types
Attendance    int64
Student_ID      str
dtype: object

Test 1
----------------------------------------
Rows: 57
Columns: 11

Missing Values
ENGLISH LANGUAGE           0
SOCIAL STUDIES             0
REL. & MORAL EDU.          0
MATHEMATICS                0
INTEGRATED SCIENCE         0
COMPUTING                  0
FRENCH                     0
GH. LANG. (AKUAPEM TWI)    0
CAREER TECHNOLOGY          0
CREATIVE ARTS & DESIGN     0
Student_ID                 0
dtype: int64

Duplicate Rows
0

Data Types
ENGLISH LANGUAGE           float64
SOCIAL STUDIES             float64
REL. & MORAL EDU.          float64
MATHEMATICS                float64
INTEGRATED SCIENCE         float64
COMPUTING                  float64
FRENCH                     float64
GH. LANG. (AKUAPEM TWI)    float64
CAREER TECHNOLOGY          float64
CREATIVE ARTS & DES

In [23]:
attendance_term2.to_csv("../data/cleaned/term2/attendance.csv", index=False)

test1_term2.to_csv("../data/cleaned/term2/test1.csv", index=False)

test2_term2.to_csv("../data/cleaned/term2/test2.csv", index=False)

groupwork_term2.to_csv("../data/cleaned/term2/groupwork.csv", index=False)

projectwork_term2.to_csv("../data/cleaned/term2/projectwork.csv", index=False)

classscore_term2.to_csv("../data/cleaned/term2/classscore.csv", index=False)

examscore_term2.to_csv("../data/cleaned/term2/examscore.csv", index=False)

print("Term 2 cleaned datasets saved successfully.")

Term 2 cleaned datasets saved successfully.


### Term 3 Data Cleaning

The data cleaning procedures applied to the first and second academic terms were repeated for the third term to maintain consistency across all datasets.

The following cleaning operations were performed:

- Removal of blank rows
- Handling of missing values
- Correction of data types
- Removal of duplicate records
- Validation of the cleaned datasets

Applying the same cleaning workflow across all three terms ensures that the datasets are standardized and ready for database integration.

In [24]:
attendance_term3 = pd.read_csv("../data/processed/term3/attendance.csv")

test1_term3 = pd.read_csv("../data/processed/term3/test1.csv")

test2_term3 = pd.read_csv("../data/processed/term3/test2.csv")

groupwork_term3 = pd.read_csv("../data/processed/term3/groupwork.csv")

projectwork_term3 = pd.read_csv("../data/processed/term3/projectwork.csv")

classscore_term3 = pd.read_csv("../data/processed/term3/classscore.csv")

examscore_term3 = pd.read_csv("../data/processed/term3/examscore.csv")

In [25]:
attendance_term3 = remove_blank_rows(attendance_term3)

test1_term3 = remove_blank_rows(test1_term3)

test2_term3 = remove_blank_rows(test2_term3)

groupwork_term3 = remove_blank_rows(groupwork_term3)

projectwork_term3 = remove_blank_rows(projectwork_term3)

classscore_term3 = remove_blank_rows(classscore_term3)

examscore_term3 = remove_blank_rows(examscore_term3)

In [26]:
attendance_term3["Attendance"] = (
    attendance_term3["Attendance"]
    .fillna(round(attendance_term3["Attendance"].mean()))
    .astype(int)
)

In [27]:
test1_term3 = fill_missing_scores(test1_term3)

test2_term3 = fill_missing_scores(test2_term3)

groupwork_term3 = fill_missing_scores(groupwork_term3)

projectwork_term3 = fill_missing_scores(projectwork_term3)

classscore_term3 = fill_missing_scores(classscore_term3)

examscore_term3 = fill_missing_scores(examscore_term3)

In [28]:
attendance_term3 = remove_duplicates(attendance_term3)

test1_term3 = remove_duplicates(test1_term3)

test2_term3 = remove_duplicates(test2_term3)

groupwork_term3 = remove_duplicates(groupwork_term3)

projectwork_term3 = remove_duplicates(projectwork_term3)

classscore_term3 = remove_duplicates(classscore_term3)

examscore_term3 = remove_duplicates(examscore_term3)

In [29]:
validate_dataset(attendance_term3, "Attendance")

validate_dataset(test1_term3, "Test 1")

validate_dataset(test2_term3, "Test 2")

validate_dataset(groupwork_term3, "Group Work")

validate_dataset(projectwork_term3, "Project Work")

validate_dataset(classscore_term3, "Class Score")

validate_dataset(examscore_term3, "Exam Score")


Attendance
----------------------------------------
Rows: 53
Columns: 2

Missing Values
Attendance    0
Student_ID    0
dtype: int64

Duplicate Rows
0

Data Types
Attendance    int64
Student_ID      str
dtype: object

Test 1
----------------------------------------
Rows: 53
Columns: 11

Missing Values
ENGLISH LANGUAGE           0
SOCIAL STUDIES             0
RME                        0
MATHEMATICS                0
INTEGRATED SCIENCE         0
COMPUTING                  0
FRENCH                     0
GH. LANG. (AKUAPEM TWI)    0
CAREER TECHNOLOGY          0
CREATIVE ARTS & DESIGN     0
Student_ID                 0
dtype: int64

Duplicate Rows
0

Data Types
ENGLISH LANGUAGE           float64
SOCIAL STUDIES             float64
RME                        float64
MATHEMATICS                float64
INTEGRATED SCIENCE         float64
COMPUTING                  float64
FRENCH                     float64
GH. LANG. (AKUAPEM TWI)    float64
CAREER TECHNOLOGY          float64
CREATIVE ARTS & DES

In [31]:
attendance_term3.to_csv("../data/cleaned/term3/attendance.csv", index=False)

test1_term3.to_csv("../data/cleaned/term3/test1.csv", index=False)

test2_term3.to_csv("../data/cleaned/term3/test2.csv", index=False)

groupwork_term3.to_csv("../data/cleaned/term3/groupwork.csv", index=False)

projectwork_term3.to_csv("../data/cleaned/term3/projectwork.csv", index=False)

classscore_term3.to_csv("../data/cleaned/term3/classscore.csv", index=False)

examscore_term3.to_csv("../data/cleaned/term3/examscore.csv", index=False)

print("Term 3 cleaned datasets saved successfully.")

Term 3 cleaned datasets saved successfully.


### Removing Derived Assessment Columns

The Exam Score dataset contained additional columns representing the 50% contribution of each subject. These values were calculated from the raw examination scores and therefore represent derived data rather than original observations.

To maintain a normalized database structure and avoid storing redundant information, the derived 50% columns were removed during the data cleaning stage. The raw examination scores were retained, allowing all totals, percentages, and other aggregate measures to be calculated later using SQL queries, Power BI measures, or Python analysis.

# Remove all derived 50% columns from the Exam Score dataset

examscore = examscore.loc[
    :,
    ~examscore.columns.str.startswith("50")
]

# Verify the remaining columns
examscore.columns

### Conclusion

All datasets for the three academic terms were successfully cleaned and standardized. Data quality issues, including blank records, missing values, inconsistent data types, and duplicate records, were addressed using a consistent cleaning workflow.

The cleaned datasets have been saved and are now ready for loading into the MySQL database in the next phase of the project.